# 지우 원본 dev26＋GMM의 직접 지도학습 수정
정상/가스 라벨별GMM 밀도분류. LR/RF 및이전7개앙상블을사용하지않는다.
기록단위·인과편차·전처리는최신지우코드를사용하고소수양성공분산만보정한다.


In [1]:
import nbformat,json,hashlib
from pathlib import Path
import numpy as np,pandas as pd
from scipy.special import expit,logsumexp
from sklearn.mixture import GaussianMixture
from sklearn.covariance import LedoitWolf
from sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve
ROOT=Path.cwd();OUT=ROOT/"reports/tables"
nb=nbformat.read("notebooks/184_cn7_source_aligned_gmm_supervised_head.ipynb",as_version=4)
old="\n".join(c.source for c in nb.cells if c.cell_type=="code")
# Only original Jiwoo data cleanup/dev26 helpers; do NOT fit/use its unsupervised teacher or any LR/RF.
exec(old[:old.index("# Original full-covariance GMM2")])
reference_n=len(ref);reference_last=ref.TimeStamp.max()
MODELS=["GMM_N1_G1","GMM_N2_G1","GMM_N2_G2"]
featurememo={}
def features(ctx):
 key=tuple(np.flatnonzero(ctx))
 if key in featurememo:return featurememo[key]
 ids=np.flatnonzero(ctx)
 # Keep real labeled Run IDs. Removing a validation block is not a physical restart.
 # Query sensor records are absent, but older accepted baseline may persist in same Run.
 value=ext26(d.iloc[ids].copy())[cols].to_numpy(float)
 out=np.full((N,26),np.nan);out[ids]=value
 featurememo[key]=out;return out
def train_context(parent,q):
 ctx=parent.copy()
 for run in pd.unique(g[q]):
  ids=q[g[q]==run]
  ctx[(g==run)&(seq>=seq[ids].min()-10)&(seq<=seq[ids].max()+10)]=False
 return ctx
def density(mean,cov,x):
 sign,ld=np.linalg.slogdet(cov)
 assert sign>0
 err=x-mean;inv=np.linalg.pinv(cov)
 return -.5*(len(mean)*np.log(2*np.pi)+ld+np.einsum("ij,jk,ik->i",err,inv,err))
def fit_models(x,yy):
 normal=yy==0;gas=yy==1
 prep=peer_models.Prep().fit(x[normal])
 a=prep(x)
 bundle={"prep":prep,"gas_count":int(gas.sum())}
 if not gas.any():
  bundle["empty_gas"]=True;return bundle
 # Same full covariance family/reg_covar/n_init/seed as Jiwoo GMM.
 bundle["normal1"]=peer_models.GMM(1,seed=0).fit(a[normal]).m
 bundle["normal2"]=peer_models.GMM(2,seed=0).fit(a[normal]).m
 prior_cov=LedoitWolf().fit(a[normal]).covariance_+np.eye(a.shape[1])*1e-3
 gas_models={}
 for requested in [1,2]:
  k=requested if gas.sum()>=max(4,requested*2) else 1
  if gas.sum()==1:
   weights=np.ones(1);means=a[gas].copy();covs=prior_cov[None,:,:]
   effective=np.ones(1)
  else:
   m=GaussianMixture(n_components=k,covariance_type="full",reg_covar=1e-3,n_init=2,random_state=0).fit(a[gas])
   weights=m.weights_;means=m.means_
   effective=m.predict_proba(a[gas]).sum(axis=0)
   covs=[]
   for n,c in zip(effective,m.covariances_):
    # 26-dimensional gas covariance cannot be trusted from1~13 samples.
    weight=n/(n+26.)
    covs.append(weight*c+(1-weight)*prior_cov+np.eye(a.shape[1])*1e-3)
   covs=np.array(covs)
  gas_models[requested]={"weights":weights,"means":means,"covariances":covs,"fitted_components":k}
 bundle["gas_models"]=gas_models
 bundle["log_prior_odds"]=np.log((gas.sum()+.5)/(normal.sum()+.5))
 return bundle
def predict_scores(bundle,x):
 if bundle.get("empty_gas"):return {n:np.full(len(x),-1e12) for n in MODELS}
 a=bundle["prep"](x);out={}
 for name,kn,kg in [("GMM_N1_G1",1,1),("GMM_N2_G1",2,1),("GMM_N2_G2",2,2)]:
  ng=bundle["normal"+str(kn)]
  gg=bundle["gas_models"][kg]
  loggas=logsumexp(np.column_stack([np.log(w)+density(m,c,a) for w,m,c in zip(gg["weights"],gg["means"],gg["covariances"])]),axis=1)
  # Use log odds directly; sigmoid rounds distinct extreme scores to1.0.
  out[name]=loggas-ng.score_samples(a)+bundle["log_prior_odds"]
 return out
memo={};audit=[]
def predict(parent,q):
 key=(tuple(np.flatnonzero(parent)),tuple(q))
 if key in memo:return memo[key]
 ctx=train_context(parent,q);ids=np.flatnonzero(ctx&eligible)
 assert not ctx[q].any() and set(ids).isdisjoint(q)
 bundle=fit_models(features(ctx)[ids],y[ids])
 ans=predict_scores(bundle,features(parent)[q])
 audit.append({"train":len(ids),"train_gas":int(y[ids].sum()),"query":len(q),
 "timestamp_overlap":len(set(d.TimeStamp.iloc[ids])&set(d.TimeStamp.iloc[q])),
 "test_raw_in_training_history":int(ctx[q].sum()),
 "covariance_type":"full","gas_one_sample_fallback":int(y[ids].sum()==1)})
 memo[key]=(ans,ids,ctx)
 return memo[key]
def stats(yy,p):
 tn,fp,fn,tp=confusion_matrix(yy,p,labels=[0,1]).ravel()
 return {"TP":int(tp),"FP":int(fp),"FN":int(fn),"TN":int(tn),
 "F1":float(2*tp/(2*tp+fp+fn)) if tp else 0,
 "precision":float(tp/(tp+fp)) if tp+fp else 0,"recall":float(tp/(tp+fn)) if tp+fn else 0}
def cutoff(yy,ss):
 _,_,values=precision_recall_curve(yy,ss);opts=[]
 for v in values:
  met=stats(yy,ss>=v)
  if met["FP"]<=int(np.floor((yy==0).sum()*.03)):
   opts.append((met["F1"],met["TP"],-met["FP"],float(v)))
 return max(opts)[-1] if opts else float(np.nextafter(ss.max(),np.inf))
records=[];choices=[]
for held in pd.unique(blocks):
 q=np.flatnonzero(blocks==held)
 outer,ids,ctx=predict(np.ones(N,bool),q)
 inner={n:np.full(len(ids),np.nan) for n in MODELS}
 for ib in pd.unique(innerblocks[ids]):
  mask=innerblocks[ids]==ib
  scores,_,_=predict(ctx,ids[mask])
  for n in MODELS:inner[n][mask]=scores[n]
 options=[]
 for n in MODELS:
  cut=cutoff(y[ids],inner[n]);met=stats(y[ids],inner[n]>=cut)
  options.append((met["F1"],met["TP"],-met["FP"],n,cut))
  for j,i in enumerate(q):
   records.append({"model":n,"TimeStamp":d.TimeStamp.iloc[i],"heldout_block":held,
    "actual":int(y[i]),"eligible":int(eligible[i]),"score":float(outer[n][j]),
    "threshold":cut,"prediction":int(outer[n][j]>=cut),"train_gas":int(y[ids].sum())})
 best=max(options);n,cut=best[-2:]
 choices.append({"heldout_block":held,"selected":n,"threshold":cut,"inner_F1":best[0],"train_gas":int(y[ids].sum())})
 for j,i in enumerate(q):
  records.append({"model":"primary_inner_selected","TimeStamp":d.TimeStamp.iloc[i],"heldout_block":held,
   "actual":int(y[i]),"eligible":int(eligible[i]),"score":float(outer[n][j]),
   "threshold":cut,"prediction":int(outer[n][j]>=cut),"train_gas":int(y[ids].sum())})
 pd.DataFrame(choices).to_csv(OUT/"cn7_185_supervised_class_gmm_progress.csv",index=False)
 print(f"Jiwoo dev26 supervised class GMM {len(choices)}/20",flush=True)
pp=pd.DataFrame(records);rows=[]
for n,a in pp.groupby("model",sort=False):
 a=a[a.eligible.eq(1)]
 rows.append({"model":n,"N":len(a),"gas":int(a.actual.sum()),**stats(a.actual,a.prediction),"AP":average_precision_score(a.actual,a.score)})
mm=pd.DataFrame(rows)
pp.to_csv(OUT/"cn7_185_supervised_class_gmm_predictions.csv",index=False)
mm.to_csv(OUT/"cn7_185_supervised_class_gmm_metrics.csv",index=False)
pd.DataFrame(audit).to_csv(OUT/"cn7_185_supervised_class_gmm_audit.csv",index=False)
pd.DataFrame(choices).to_csv(OUT/"cn7_185_supervised_class_gmm_choices.csv",index=False)
assert pd.DataFrame(audit).timestamp_overlap.eq(0).all() and pd.DataFrame(audit).test_raw_in_training_history.eq(0).all()
# Full-label final fit, selected and calibrated by training-only crossfit.
ids=np.flatnonzero(eligible);parent=np.ones(N,bool)
inner={n:np.full(len(ids),np.nan) for n in MODELS}
for ib in pd.unique(innerblocks[ids]):
 mask=innerblocks[ids]==ib;scores,_,_=predict(parent,ids[mask])
 for n in MODELS:inner[n][mask]=scores[n]
opts=[];thresholds={}
for n in MODELS:
 cut=cutoff(y[ids],inner[n]);thresholds[n]=cut
 met=stats(y[ids],inner[n]>=cut);opts.append((met["F1"],met["TP"],-met["FP"],n,cut))
chosen=max(opts)[-2]
final=fit_models(features(parent)[ids],y[ids])
us=predict_scores(final,F[cols].to_numpy(float))
ur=[]
for n in MODELS:
 pred=us[n]>=thresholds[n]
 F["prediction_"+n]=pred.astype(int)
 ur.append({"model":n,"selected":int(n==chosen),"clean_unknown":len(F),"alarms":int(pred.sum()),
 "alarm_percent":float(100*pred.mean()),"threshold":thresholds[n]})
um=pd.DataFrame(ur)
F[["TimeStamp","run_id"]+["prediction_"+n for n in MODELS]].to_csv(OUT/"cn7_185_supervised_class_gmm_unlabeled_predictions.csv",index=False)
um.to_csv(OUT/"cn7_185_supervised_class_gmm_unlabeled_metrics.csv",index=False)
p=pp[pp.model.eq("primary_inner_selected")].merge(d[["TimeStamp","gas","any_fail"]],on="TimeStamp",validate="one_to_one")
p[p.gas].to_csv(OUT/"cn7_185_supervised_class_gmm_all13_gas.csv",index=False)
prov={"source_branch":"origin/jiwoo f02c9fd","source_common_sha256":hashlib.sha256(cs.encode()).hexdigest(),
 "features":cols,"preprocessing":"Jiwoo Prep + exactdev26 including1e-3relation units",
 "supervision":"normal and gas labeled-class separate full-cov GMM; likelihood-ratio Bayes classifier",
 "no_LR_RF_or_old_ensemble":True,"gas_covariance_shrinkage":"n/(n+26) sample covariance versus normal LedoitWolf prior; single labeled gas Gaussian fallback, no duplicated fake labels",
 "validation":"same20 temporal blocks/purge10/nested5; retain physicalRun IDs, omit query rawrecords from training history",
 "selection":"innerF1 with normalFP3pct budget; no outer-label cutoff fitting",
 "unlabeled":"sameJiwoo cleaned8007shots; not old28pct cohort; no unlabeled labels used for fitting",
 "caveat":"same-run temporal development, not forward/unseenRun independent validation"}
(OUT/"cn7_185_supervised_class_gmm_provenance.json").write_text(json.dumps(prov,ensure_ascii=False,indent=2),encoding="utf-8")
print("\nSUPERVISED LABEL RESULTS\n",mm.to_string(index=False))
print("\nSUPERVISED UNLABELED RESULTS\n",um.to_string(index=False))
print("\nPRIMARY GAS\n",p[p.gas][["TimeStamp","score","threshold","prediction","train_gas"]].to_string(index=False))
lines=["# 지우dev26/GMM을직접지도학습으로수정한결과","",
 "이전7개앙상블과28%비라벨분석은사용하지않는다. LR/RF도붙이지않았다.",
 "원본dev26/Prep/전체공분산GMM을사용하고정상라벨/가스라벨별분포를학습해두밀도의비로분류한다.",
 "기본정상GMM2/가스GMM1, 비교정상1/가스1 및정상2/가스2. 가스1개구간에서는사전공분산으로보정한단일가우시안fallback이며가짜라벨을복제하지않는다.",
 "가스가26차원공분산을추정하기에적어정상LW공분산으로shrinkage를추가했다. 이부분은지도학습용수정이다.",
 "물리적Run은유지하며평가구간및앞뒤10샷원시기록을학습이력에서제거한다. 검증제외를인위적인공정재시작으로처리하지않는다.",
 "같은605샷/20시간블록/purge10/nested5, 모델/임계값선택은안쪽검증만사용한다.",
 "결측은지우Prep처럼0으로처리한다. 비라벨은지우정제후8007샷으로이전집계분모와다르다.",
 "정제비라벨정답은없다. 경보율이낮아도실제정확도라고해석하지않는다.","",
 "## 라벨",mm.to_string(index=False),"","## 비라벨",um.to_string(index=False),"",
 "## 주모델의가스13개",p[p.gas][["TimeStamp","score","threshold","prediction","train_gas"]].to_string(index=False),
 "","독립새Run/미래검증이아니며이미본자료에서의개발조건이다. 다른모델을몰래섞거나평가범위를변경하지않았다."]
(ROOT/"reports/cn7_185_jiwoo_supervised_class_gmm.md").write_text("\n".join(lines),encoding="utf-8")


Jiwoo dev26 supervised class GMM 1/20


Jiwoo dev26 supervised class GMM 2/20


Jiwoo dev26 supervised class GMM 3/20


Jiwoo dev26 supervised class GMM 4/20


Jiwoo dev26 supervised class GMM 5/20


...

Jiwoo dev26 supervised class GMM 18/20


Jiwoo dev26 supervised class GMM 19/20


Jiwoo dev26 supervised class GMM 20/20



SUPERVISED LABEL RESULTS
                  model   N  gas  TP  FP  FN  TN  F1  precision  recall       AP
             GMM_N1_G1 605   13   0   2  13 590   0        0.0     0.0 0.109741
             GMM_N2_G1 605   13   0   3  13 589   0        0.0     0.0 0.179039
             GMM_N2_G2 605   13   0   0  13 592   0        0.0     0.0 0.175809
primary_inner_selected 605   13   0   3  13 589   0        0.0     0.0 0.211387

SUPERVISED UNLABELED RESULTS
     model  selected  clean_unknown  alarms  alarm_percent  threshold
GMM_N1_G1         0           8007     174       2.173099   9.207576
GMM_N2_G1         1           8007     269       3.359560  14.989402
GMM_N2_G2         0           8007     309       3.859123  14.989402

PRIMARY GAS
           TimeStamp      score   threshold  prediction  train_gas
2020-10-16 05:21:38 -10.365112  735.347510           0         11
2020-10-16 05:37:29 -24.661419 1590.090870           0          1
2020-10-16 05:45:26  14.951441 1590.090870           0

2148